# 03b - Rekayasa Fitur & Transformasi Variabel
Materi Modul 3: transformasi variabel, pengkodean fitur, penskalaan fitur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures

df = pd.read_csv('../datasets/wine.csv')

## 1. Transformasi variabel (mengatasi skewness)

In [ ]:
kolom = 'malic_acid'
x = df[kolom]
cand = {
    'Asli': x,
    'log1p': np.log1p(x),
    'sqrt': np.sqrt(x),
    'Yeo-Johnson': pd.Series(PowerTransformer().fit_transform(x.to_frame()).ravel()),
}
fig, ax = plt.subplots(1, 4, figsize=(16,3.5))
for a, (n, v) in zip(ax, cand.items()):
    a.hist(v, bins=20); a.set_title(f'{n} (skew={v.skew():.2f})')
plt.tight_layout(); plt.show()

## 2. Pengkodean fitur kategorikal
Wine hanya punya fitur numerik, jadi kita buat fitur kategorikal dari `alcohol` lewat binning.

In [ ]:
df['kadar_alkohol'] = pd.qcut(df['alcohol'], q=3, labels=['rendah','sedang','tinggi'])
df[['alcohol','kadar_alkohol']].head()

In [ ]:
# One-hot encoding (tanpa urutan)
pd.get_dummies(df['kadar_alkohol'], prefix='alk').head()

In [ ]:
# Ordinal encoding (ada urutan rendah < sedang < tinggi)
oe = OrdinalEncoder(categories=[['rendah','sedang','tinggi']])
df['alk_ordinal'] = oe.fit_transform(df[['kadar_alkohol']]).astype(int)
# Label encoding untuk target berupa teks
le = LabelEncoder()
print(le.fit_transform(['a','b','c','a']), le.classes_)
df[['kadar_alkohol','alk_ordinal']].drop_duplicates().sort_values('alk_ordinal')

## 3. Penskalaan fitur

In [ ]:
fit = ['alcohol','proline']
skalers = {'Asli': None, 'Standard': StandardScaler(), 'MinMax': MinMaxScaler(), 'Robust': RobustScaler()}
fig, ax = plt.subplots(1, 4, figsize=(16,3.5))
for a, (n, s) in zip(ax, skalers.items()):
    d = df[fit] if s is None else pd.DataFrame(s.fit_transform(df[fit]), columns=fit)
    d.boxplot(ax=a); a.set_title(n)
plt.tight_layout(); plt.show()

## 4. Fitur interaksi / polinomial

In [ ]:
pf = PolynomialFeatures(degree=2, include_bias=False)
P = pf.fit_transform(df[['alcohol','flavanoids']])
pd.DataFrame(P, columns=pf.get_feature_names_out()).head()

## Catatan
- (kapan pakai log vs sqrt, one-hot vs ordinal, Standard vs MinMax vs Robust)